# Milestone 2: Hierarchical Parsing, Standardization, and Deduplication Pipeline

Notebook này triển khai toàn bộ luồng xử lý tự động cho **Milestone 2**: chuyển đổi mã nguồn thô LaTeX (`.tex`, `.bib`, `.bbl`) thu thập từ Milestone 1 thành tập dữ liệu phân cấp, chuẩn hóa và khử trùng lặp theo **Data Schema v2**.

### 🎯 Kiến trúc luồng xử lý khép kín trên RAM theo từng `arxiv_id`:
1. **Tối ưu I/O cục bộ trên Colab:** Giải nén `raw_data.zip` về ổ SSD `/content/local_raw` để tăng tốc độ đọc file TeX gấp nhiều lần so với đọc trực tiếp qua Google Drive FUSE.
2. **Xử lý khép kín trên RAM:**
   - **LaTeX AST Parser (`src.parser_module.parse_tex_versions`):** Tự động phát hiện file root, giải quyết transitive `\input`/`\include`, chuẩn hóa toán (`$ ... $`), dựng cây phân cấp (`Document` $\rightarrow$ `Section` $\rightarrow$ `Paragraph` $\rightarrow$ `Sentence` / `Math` / `Figure`).
   - **BibTeX Parser (`src.parser_module.bib_parser`):** Bóc tách tài liệu tham khảo từ `.bib` hoặc inline `\bibitem`, chuẩn hóa các trường metadata.
   - **Deduplicator (`src.deduplicator.deduplicate_paper`):** Khử trùng lặp nội dung văn bản `elements` và quan hệ `hierarchy` giữa các phiên bản, hợp nhất thông tin các mục `bibtex`.
3. **Batch Flushing & Parquet Output:** Xả dữ liệu ra các file `.parquet` nén định kỳ theo từng lô $N$ bài báo, giải phóng bộ nhớ RAM (`gc.collect()`), tránh tràn RAM trên Colab CPU-only.
4. **System Benchmark:** Tích hợp `SystemBenchmarkTracker` theo dõi Wall time, Peak RAM, Average RAM, Peak Disk và Dung lượng Parquet cuối cùng.

## 1. Kết nối Google Drive & Thiết lập môi trường SSD cục bộ

Để tốc độ xử lý nhanh nhất, chúng ta sao chép file nén dữ liệu thô `raw_data.zip` từ Google Drive sang ổ SSD cục bộ của máy ảo Colab (`/content/`) và giải nén tại chỗ.

In [ ]:
# 1.1. Kết nối Google Drive và đồng bộ mã nguồn mới nhất từ GitHub
import os
import subprocess
import sys
from pathlib import Path

# Nhánh Git hiện tại của Milestone 2
BRANCH = "feat/m2-pipeline"

try:
    from google.colab import drive
except ImportError:
    CODE_DIR = Path.cwd().resolve()
    print(f"Đang chạy ở môi trường cục bộ (không phải Google Colab). CODE_DIR: {CODE_DIR}")
else:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    
    if not (CODE_DIR / ".git").is_dir():
        print(f"--> Đang clone mã nguồn từ nhánh {BRANCH} về {CODE_DIR}...")
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        print(f"--> Đang cập nhật (pull) mã nguồn mới nhất từ nhánh {BRANCH}...")
        subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(CODE_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "origin", BRANCH], check=True)

    # Chuyển working directory sang thư mục mã nguồn
    os.chdir(CODE_DIR)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("Thư mục mã nguồn:", CODE_DIR, "| Nhánh Git:", BRANCH)


### 1.2. Sao chép và giải nén dữ liệu thô sang ổ SSD cục bộ

> **Lưu ý:** Bạn hãy điều chỉnh đường dẫn file `raw_data.zip` trên Drive của bạn nếu khác với đường dẫn mặc định.

In [ ]:
#@title Đường dẫn file nén dữ liệu thô trên Google Drive { display-mode: "form" }
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/raw_data.zip" #@param {type:"string"}

import os
import shutil
from pathlib import Path

# Tạo thư mục đích trên SSD cục bộ
!mkdir -p /content/local_raw /content/local_processed

if Path(DRIVE_ZIP_PATH).is_file():
    print(f"--> Đang sao chép {DRIVE_ZIP_PATH} sang /content/...")
    !cp "$DRIVE_ZIP_PATH" /content/
    zip_name = Path(DRIVE_ZIP_PATH).name
    print(f"--> Đang giải nén /content/{zip_name} vào /content/local_raw (chế độ im lặng -q)...")
    !unzip -q "/content/$zip_name" -d /content/local_raw
    print("Giải nén hoàn tất!")
    
    # Chuẩn hóa nếu file zip chứa sẵn thư mục con 'raw' (ví dụ /content/local_raw/raw/...)
    nested_raw = Path("/content/local_raw/raw")
    if nested_raw.is_dir():
        print("⚡ Phát hiện thư mục lồng /content/local_raw/raw, tự động đưa các bài báo ra /content/local_raw/...")
        !mv /content/local_raw/raw/* /content/local_raw/ 2>/dev/null || true
        !rmdir /content/local_raw/raw 2>/dev/null || true
else:
    print(f"Chưa tìm thấy {DRIVE_ZIP_PATH}.")
    print("Nếu bạn đang chạy thử nghiệm với dữ liệu repo, có thể sử dụng data/raw có sẵn.")

### 1.3. Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt các thư viện phục vụ xử lý và lưu trữ Parquet
!pip install -q pyarrow fastparquet pandas psutil PyYAML tqdm matplotlib

## 2. Cấu hình tham số chạy Pipeline Milestone 2

Chọn thông tin thành viên (để lưu tên file kết quả), dải bài báo và kích thước lô xử lý (`batch_size`).

> 💡 **Vai trò của các tham số thực thi:**
> * **`BATCH_SIZE` (Cơ chế Batch Flushing):** Thay vì dồn toàn bộ dữ liệu của hàng nghìn bài báo trên RAM, cứ sau mỗi `BATCH_SIZE` bài (mặc định 50 bài), pipeline sẽ tự động ghi các mảnh `.parquet` xuống đĩa, cập nhật các file CSV và gọi `gc.collect()` để giải phóng RAM ngay lập tức, chống tràn RAM (OOM) và lưu checkpoint an toàn.
> * **`MAX_WORKERS` (Số tiến trình CPU song song):** Tận dụng đa nhân CPU (`ProcessPoolExecutor`) để xử lý đồng thời nhiều bài báo. Trên Colab CPU-only miễn phí (thường có 2 vCPU) hoặc Colab Pro (thường có 4 vCPU), khuyến nghị đặt từ **2 đến 4** workers để đạt tốc độ tối đa mà không gây quá tải tài nguyên.

In [ ]:
#@title Cấu hình tham số Pipeline { display-mode: "form" }
#@markdown ### 1. Thông tin thành viên & Thư mục làm việc
MEMBER_NAME = "Dac_Anh" #@param ["Dac_Anh", "Hao", "Toan", "Tuan", "Duy"] {allow-input: true}
RAW_DIR = "/content/local_raw" #@param {type:"string"}
PROCESSED_DIR = "/content/local_processed" #@param {type:"string"}

# Tự động fallback về thư mục data/raw trong repository nếu không chạy trên Colab
if not Path(RAW_DIR).exists() and Path("data/raw").exists():
    RAW_DIR = "data/raw"
    PROCESSED_DIR = "data/processed"
    print(f"Fallback về đường dẫn repo: RAW_DIR={RAW_DIR}, PROCESSED_DIR={PROCESSED_DIR}")

# Tự động nhận diện thư mục lồng 'raw' hoặc 'data/raw' sau khi giải nén
raw_check = Path(RAW_DIR)
if (raw_check / "raw").is_dir() and not any(f.startswith("v") for f in os.listdir(raw_check) if (raw_check / f).is_dir()):
    RAW_DIR = str(raw_check / "raw")
    print(f"Tự động cập nhật RAW_DIR trỏ vào thư mục lồng: {RAW_DIR}")
elif (raw_check / "data" / "raw").is_dir():
    RAW_DIR = str(raw_check / "data" / "raw")
    print(f"Tự động cập nhật RAW_DIR trỏ vào thư mục lồng: {RAW_DIR}")

#@markdown ### 2. Tùy chọn thực thi đa luồng & Batch Flushing
#@markdown * **BATCH_SIZE (Cơ chế xả đĩa & dọn RAM):** Số lượng bài báo gom trong một lô xử lý. Cứ hoàn thành mỗi `BATCH_SIZE` bài, pipeline sẽ tự động xả các mảnh Parquet xuống đĩa, lưu checkpoint vào 3 file CSV thống kê và gọi `gc.collect()` giải phóng RAM ngay lập tức để tránh lỗi tràn bộ nhớ (Out-Of-Memory) trên Colab.
BATCH_SIZE = 50 #@param {type:"integer"}
#@markdown * **MAX_WORKERS (Số tiến trình CPU song song):** Số lượng worker CPU chạy đa tiến trình (`ProcessPoolExecutor`). Khuyến nghị đặt 2 trên Colab CPU-only (2 vCPUs) để tối ưu hóa hiệu năng, tránh tranh chấp CPU và deadlock.
MAX_WORKERS = 2 #@param {type:"integer"}
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/m2_processed" #@param {type:"string"}

#@markdown ### 3. Giới hạn số lượng bài báo (nhập 0 để xử lý tất cả bài có trong RAW_DIR)
LIMIT_PAPERS = 0 #@param {type:"integer"}

print(f"Cấu hình: Member={MEMBER_NAME} | BatchSize={BATCH_SIZE} | Workers={MAX_WORKERS} | Limit={LIMIT_PAPERS or 'ALL'}")

## 3. Khảo sát & Kiểm tra danh sách bài báo đầu vào

In [ ]:
import sys
from pathlib import Path

# Thêm đường dẫn project vào sys.path để import các module src
current_dir = Path.cwd()
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

from src.scripts.run_m2_pipeline import find_paper_folders

raw_path = Path(RAW_DIR)
if raw_path.exists():
    all_paper_folders = find_paper_folders(raw_path)
    total_found = len(all_paper_folders)
    print(f"✅ Tìm thấy {total_found:,} thư mục bài báo hợp lệ trong {raw_path}.")
    if total_found > 0:
        print("Mẫu 5 bài báo đầu tiên:", [d.name for d in all_paper_folders[:5]])
    else:
        print("⚠️ Cảnh báo: Không tìm thấy thư mục bài báo nào trong {raw_path}!")
else:
    print(f"Cảnh báo: Thư mục {raw_path} chưa tồn tại!")

## 4. Thực thi Pipeline khép kín trên RAM

Chạy module `src.scripts.run_m2_pipeline`: 
- Tận dụng đa nhân CPU (`ProcessPoolExecutor`).
- Kết hợp cả 3 công đoạn: AST Parser $\rightarrow$ BibTeX Parser $\rightarrow$ Deduplication hoàn toàn trên RAM.
- Tự động xuất `.parquet` và ghi các file CSV chỉ số theo từng lô $N$ bài báo (Batch Flushing).
- Khởi động `SystemBenchmarkTracker` theo dõi tài nguyên phần cứng.

In [ ]:
from src.scripts.run_m2_pipeline import run_m2_pipeline

# Chọn dải bài báo cần chạy (hoặc toàn bộ)
target_ids = None
if LIMIT_PAPERS > 0:
    target_ids = [d.name for d in all_paper_folders[:LIMIT_PAPERS]]
    print(f"Đã giới hạn xử lý {len(target_ids)} bài báo đầu tiên để kiểm thử.")
else:
    print(f"Tiến hành xử lý toàn bộ {len(all_paper_folders)} bài báo trong {RAW_DIR}...")

# Hàm cập nhật tiến độ
def print_progress(done, total, cur_id):
    if done % 10 == 0 or done == total:
        pct = (done / total) * 100
        print(f"[{done}/{total} - {pct:.1f}%] Đã xử lý xong: {cur_id}")

# Bắt đầu thực thi pipeline
results = run_m2_pipeline(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    member_name=MEMBER_NAME,
    paper_ids=target_ids,
    batch_size=BATCH_SIZE,
    max_workers=MAX_WORKERS,
    progress_callback=print_progress,
)

print("\nPipeline Milestone 2 đã hoàn thành xuất sắc!")

## 5. Tổng kết kết quả & Kiểm định 4 Bảng chỉ số

Hiển thị chi tiết số liệu từ 4 bảng CSV và các file thành phẩm `.parquet`:

In [ ]:
import pandas as pd
from IPython.display import display

proc_dir = Path(PROCESSED_DIR)
tree_csv = Path(results["tree_stats_csv"])
bib_csv = Path(results["bib_stats_csv"])
dedup_csv = Path(results["dedup_stats_csv"])
bench_csv = Path(results["benchmark_csv"])

print("=" * 80)
print("1. BẢNG 1: THỐNG KÊ CÂY PHÂN CẤP (m2_tree_stats)")
print("=" * 80)
if tree_csv.exists():
    df_tree = pd.read_csv(tree_csv)
    print(f"Tổng số bài báo: {len(df_tree)} | Parse thành công: {(df_tree['parse_status'] == 'success').sum()}")
    print(f"Độ sâu cây trung bình: {df_tree['max_tree_depth'].mean():.2f} (Max: {df_tree['max_tree_depth'].max()})")
    display(df_tree.head(5))

print("\n" + "=" * 80)
print("2. BẢNG 2: THỐNG KÊ BÓC TÁCH BIBTEX (m2_bib_stats)")
print("=" * 80)
if bib_csv.exists():
    df_bib = pd.read_csv(bib_csv)
    print(f"Tổng số trích dẫn thô: {df_bib['raw_bib_count'].sum():,} | Nguồn trích dẫn:")
    print(df_bib['bib_source_type'].value_counts())
    display(df_bib.head(5))

print("\n" + "=" * 80)
print("3. BẢNG 3: THỐNG KÊ KHỬ TRÙNG LẶP (m2_dedup_stats)")
print("=" * 80)
if dedup_csv.exists():
    df_dedup = pd.read_csv(dedup_csv)
    avg_ratio = df_dedup['element_dedup_ratio'].mean() * 100
    print(f"Tỷ lệ khử trùng lặp trung bình: {avg_ratio:.2f}%")
    print(f"Tổng số phần tử sạch sau khử trùng: {df_dedup['dedup_elements_count'].sum():,}")
    display(df_dedup.head(5))

print("\n" + "=" * 80)
print("4. BẢNG 4: BENCHMARK TÀI NGUYÊN HỆ THỐNG (m2_system_benchmark)")
print("=" * 80)
if bench_csv.exists():
    df_bench = pd.read_csv(bench_csv)
    display(df_bench)

### 5.2. Kiểm tra tính toàn vẹn và dung lượng các file Parquet thành phẩm

In [ ]:
elem_p = Path(results["elements_parquet"])
hier_p = Path(results["hierarchy_parquet"])
bib_p = Path(results.get("bibtex_parquet", results.get("bib_parquet")))

print(f"{'Bảng Parquet':<25} | {'Dung lượng (KB)':<18} | {'Số dòng (Rows)':<15}")
print("-" * 65)
for name, p in [("elements.parquet", elem_p), ("hierarchy.parquet", hier_p), ("bibtex.parquet", bib_p)]:
    if p.exists():
        size_kb = p.stat().st_size / 1024
        df = pd.read_parquet(p)
        print(f"{name:<25} | {size_kb:<18,.1f} | {len(df):<15,}")
    else:
        print(f"{name:<25} | Chưa tìm thấy!")


### 5.3. Trực quan hóa số liệu phân tích nhanh (EDA)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Phân bố độ sâu cây
if tree_csv.exists() and not df_tree.empty:
    axes[0].hist(df_tree["max_tree_depth"], bins=range(1, 12), color="#2b5c8f", edgecolor="black", align="left")
    axes[0].set_title("Phân bố độ sâu cây tối đa (max_tree_depth)")
    axes[0].set_xlabel("Độ sâu cây (levels)")
    axes[0].set_ylabel("Số lượng bài báo")
    axes[0].grid(axis="y", alpha=0.3)

# Biểu đồ 2: Phân bố tỷ lệ khử trùng lặp phần tử
if dedup_csv.exists() and not df_dedup.empty:
    axes[1].hist(df_dedup["element_dedup_ratio"] * 100, bins=20, color="#27ae60", edgecolor="black")
    axes[1].set_title("Phân bố tỷ lệ nén khử trùng lặp (element_dedup_ratio)")
    axes[1].set_xlabel("Tỷ lệ trùng lặp loại bỏ (%)")
    axes[1].set_ylabel("Số lượng bài báo")
    axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Đồng bộ kết quả ngược lại Google Drive

Sau khi chạy hoàn tất toàn bộ dải bài báo được giao, copy toàn bộ thư mục `/content/local_processed` (chứa các file `.parquet` và `.csv` mang tên của bạn) lên thư mục Drive chung của nhóm.

In [ ]:
#@title Đồng bộ kết quả lên Google Drive { display-mode: "form" }
if Path("/content/drive").exists() and Path(PROCESSED_DIR).exists():
    print(f"--> Đang sao lưu {PROCESSED_DIR} lên Google Drive: {DRIVE_BACKUP_DIR}...")
    !mkdir -p "$DRIVE_BACKUP_DIR"
    !cp -rv "$PROCESSED_DIR"/* "$DRIVE_BACKUP_DIR"/
    print("Đã sao lưu toàn bộ kết quả lên Google Drive thành công!")
else:
    print("Không phát hiện Google Drive hoặc thư mục processed. Bỏ qua bước đồng bộ Drive.")